In [ ]:
!git clone https://github.com/CryAndRRich/codapath.git

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
!pip install -r requirements.txt
!pip install -U huggingface_hub hf-transfer

In [ ]:
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login

login("YOUR_HUGGINGFACE_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Đang tải vinid/plip...")
snapshot_download(repo_id="vinid/plip")

print("Đang tải PubMedBERT...")
snapshot_download(repo_id="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")

print("Đang tải BiomedCLIP...")
snapshot_download(repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

In [5]:
import sys
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [6]:
import yaml
import numpy as np
import torch

from set_up import set_seed
from load_data import get_data_loaders
from model import CODAModel, load_model
from evaluate import evaluate_model

In [7]:
PATHMNIST_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist": PATHMNIST_PATH,
    "histoset": HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH
}

In [8]:
CONFIG_PATH = "config/config.yaml"

# pathmnist, histoset, skintissue
DATASET = "histoset"

# random, coreset, codapath
# entropy, margin, badge, typiclust, activeft
SAMPLER_NAME = "random"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

hyper = config.get("hyperparameters", {})

random_seed = config["random_seed"]
device = torch.device(config["device"])

In [ ]:
seed_worker_fn = set_seed(random_seed)
g_seed = torch.Generator()
g_seed.manual_seed(random_seed)

In [ ]:
_, test_loader, class_names = get_data_loaders(DATA_DICT[DATASET], random_seed, verbose=True)
test_dataset = test_loader.dataset
test_labels = test_dataset.lbl if hasattr(test_dataset, "lbl") else np.array(test_dataset.dataset.targets)[test_dataset.indices]

In [ ]:
for budget in config["cumulative_budget"]:
    model = CODAModel(
        num_classes=len(class_names), 
        r=hyper["rank_lora"], 
        lora_alpha=hyper["rank_lora"] * 2
    ).to(device)

    checkpoint_file = f"{CODAPATH}/weights/{SAMPLER_NAME}/{DATASET}/{DATASET}_{SAMPLER_NAME}_budget_{budget}.pth"

    model = load_model(model, checkpoint_file, device)

    print(f"=== {DATASET} | {SAMPLER_NAME} | budget {budget} ===")
    print(f"Checkpoint file: {checkpoint_file}")
    evaluate_model(model, test_loader, test_labels, device)